<a href="https://colab.research.google.com/github/fciker2603-cmd/Practicas-Sistemas-Operativos-multiusuario/blob/Pr%C3%A1cticas-sistemas-operativos/Pr%C3%A1ctica_2_de_SO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import multiprocessing
import time
import random




def Productor(Memoria: multiprocessing.Array, Semaforo: multiprocessing.Semaphore, Total: int) -> None:
    """Escribe valores aleatorios en memoria compartida protegida por semáforo.


    Args:
        Memoria: Array compartido entre procesos.
        Semaforo: Sdoemáforo binario para exclusión mutua.
        Total: Cantidad de valores a escribir.
    """
    for I in range(Total):
        Valor = random.randint(1, 100)
        Semaforo.acquire()          # wait(S)
        Memoria[I] = Valor
        print(f"  Productor escribió: {Valor} en posición {I}")
        Semaforo.release()          # signal(S)
        time.sleep(0.15)

def Consumidor(Memoria: multiprocessing.Array, Semaforo: multiprocessing.Semaphore, Total: int) -> None:
    """Lee valores de memoria compartida protegida por semáforo.


    Args:
        Memoria: Array compartido entre procesos.
        Semaforo: Semáforo binario para exclusión mutua.
        Total: Cantidad de valores a leer.
    """
    for I in range(Total):
        time.sleep(0.2)
        Semaforo.acquire()          # wait(S)
        print(f"  Consumidor leyó:   {Memoria[I]} de posición {I}")
        Semaforo.release()          # signal(S)




if __name__ == "__main__":


    Total    = 5
    Memoria  = multiprocessing.Array("i", [0] * Total)
    Semaforo = multiprocessing.Semaphore(1)


    print("=" * 50)
    print("Con semáforo (acceso ordenado)")
    print("=" * 50)


    ProcProductor  = multiprocessing.Process(target=Productor,  args=(Memoria, Semaforo, Total))
    ProcConsumidor = multiprocessing.Process(target=Consumidor, args=(Memoria, Semaforo, Total))


    ProcProductor.start()
    ProcConsumidor.start()
    ProcProductor.join()
    ProcConsumidor.join()


Con semáforo (acceso ordenado)
  Productor escribió: 83 en posición 0
  Productor escribió: 45 en posición 1
  Consumidor leyó:   83 de posición 0
  Productor escribió: 37 en posición 2
  Consumidor leyó:   45 de posición 1
  Productor escribió: 70 en posición 3
  Consumidor leyó:   37 de posición 2
  Productor escribió: 25 en posición 4
  Consumidor leyó:   70 de posición 3
  Consumidor leyó:   25 de posición 4


In [ ]:
def RoundRobin(Procesos: list[dict], Quantum: int) -> None:
    """Simula Round Robin e imprime diagrama de Gantt y tiempo de espera por proceso.
    Args:
        Procesos: Lista de dicts con keys: nombre, burst.
        Quantum: Tiempo máximo de CPU por turno (ms).
    """
    Cola     = [{"nombre": P["nombre"], "restante": P["burst"]} for P in Procesos]
    Tiempo   = 0
    Espera   = {P["nombre"]: 0 for P in Procesos} #2
    Diagrama = []


    print(f"Quantum = {Quantum} ms\n{'-' * 40}")


    while Cola:
        Actual = Cola.pop(0)
        Turno  = min(Quantum, Actual["restante"])
        Sobra = Actual['restante'] - Turno
        print(f"t={Tiempo}ms — {Actual['nombre']} correrá  {Turno}ms    " f"(restante: {Actual['restante']} → {Sobra}))")


        for Esperando in Cola:
            Espera[Esperando["nombre"]] += Turno


        Tiempo             += Turno
        Actual["restante"] -= Turno
        if Actual["restante"] > 0:
            Cola.append(Actual)
        else:
            print(f"  t={Tiempo}ms — {Actual['nombre']} TERMINÓ")


    Promedio = sum(Espera.values()) / len(Espera) #2


    print(f"\n{'=' * 40}\nProceso   Espera\n{'-' * 40}")
    for Nombre, TiempoEspera in Espera.items():
        print(f"  {Nombre}   {TiempoEspera} ms")
    print(f"{'-' * 40}\n  Promedio  {Promedio:.1f} ms")




if __name__ == "__main__":


    Procesos = [
        {"nombre": "P1", "burst": 8},
        {"nombre": "P2", "burst": 3},
        {"nombre": "P3", "burst": 2}
    ]


    print("ESCENARIO 1 — Quantum pequeño (q=2)")
    print("=" * 40)
    RoundRobin([{"nombre": P["nombre"], "burst": P["burst"]} for P in Procesos], Quantum=2)


    print("\nESCENARIO 2 — Quantum grande (q=10)")
    print("=" * 40)
    RoundRobin([{"nombre": P["nombre"], "burst": P["burst"]} for P in Procesos], Quantum=10)

ESCENARIO 1 — Quantum pequeño (q=2)
Quantum = 2 ms
----------------------------------------
t=0ms — P1 correrá  2ms    (restante: 8 → 6))
t=2ms — P2 correrá  2ms    (restante: 3 → 1))
t=4ms — P3 correrá  2ms    (restante: 2 → 0))
  t=6ms — P3 TERMINÓ
t=6ms — P1 correrá  2ms    (restante: 6 → 4))
t=8ms — P2 correrá  1ms    (restante: 1 → 0))
  t=9ms — P2 TERMINÓ
t=9ms — P1 correrá  2ms    (restante: 4 → 2))
t=11ms — P1 correrá  2ms    (restante: 2 → 0))
  t=13ms — P1 TERMINÓ

Proceso   Espera
----------------------------------------
  P1   5 ms
  P2   6 ms
  P3   4 ms
----------------------------------------
  Promedio  5.0 ms

ESCENARIO 2 — Quantum grande (q=10)
Quantum = 10 ms
----------------------------------------
t=0ms — P1 correrá  8ms    (restante: 8 → 0))
  t=8ms — P1 TERMINÓ
t=8ms — P2 correrá  3ms    (restante: 3 → 0))
  t=11ms — P2 TERMINÓ
t=11ms — P3 correrá  2ms    (restante: 2 → 0))
  t=13ms — P3 TERMINÓ

Proceso   Espera
----------------------------------------
  P1   0 

In [1]:
from multiprocessing import Process
from multiprocessing import Pipe
from time import sleep

# creacion de tuberia - unidireccional
[extremo_emisor, extremo_receptor] = Pipe(False)


def receptor(extremo_tubo):
    """Tarea para la recepcion de datos"""
    print("Receptor listo")
    if extremo_tubo.poll(None) is True:
        while extremo_tubo.poll() is True:
            # recepcion - un elemento a la vez
            elemento = extremo_tubo.recv()
            print(f"recibido: {elemento}")

        print("recepcion finalizada")
    else:
        print("tuberia vacia")

    extremo_tubo.close()
    print()


def transmisor(extremo_tubo):
    """Tarea para el envio de datos"""
    print("Transmisor listo")
    lista = ["hola", 1.0 , True, 27]
    for l in lista:
        # transmision - un elemento a la vez
        extremo_tubo.send(l)
        print(f"enviado: {l}")

    print("transmision finalizada")
    extremo_tubo.close()
    print()


# subprocesos para gestionar la tuberia
sub_transmisor = Process(
    target=transmisor,
    args=(extremo_receptor,),
    daemon=True,
    )

sub_receptor = Process(
    target=receptor,
    args=(extremo_emisor,),
    daemon=True,
    )

# se carga la tuberia
sub_transmisor.start()
# lectura de datos atrasada
sleep(0.2)
sub_receptor.start()

# espera al cierre de procesos
sub_transmisor.join()
sub_receptor.join()
print("Finalizado")

Transmisor listo
enviado: hola
enviado: 1.0
enviado: True
enviado: 27
transmision finalizada

Receptor listo
recibido: hola
recibido: 1.0
recibido: True
recibido: 27
recepcion finalizada

Finalizado
